# Statistiques avec R

Tests d'hypothèses, régression et intervalles de confiance à l'aide des jeux de données intégrés de R.

Aucun téléchargement de données ni installation de paquets requis — utilise uniquement le R de base (Base R).

## 1. Statistiques descriptives

In [ ]:
data(mtcars)
cat("Jeu de données : mtcars (", nrow(mtcars), " voitures, ", ncol(mtcars), " variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. Test t pour deux échantillons

Les voitures à boîte manuelle ont-elles une meilleure consommation de carburant (MPG) que les automatiques ?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automatique :", round(mean(auto), 1), "MPG (N =", length(auto), ")\n")
cat("Manuelle: ", round(mean(manual), 1), "MPG (N =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusion :",
    ifelse(t_result$p.value < 0.05,
           "Rejet de H0 — les voitures manuelles ont un MPG significativement supérieur",
           "Échec du rejet de H0"))

## 3. Test du Chi-deux

Le nombre de cylindres et le type de transmission sont-ils indépendants ?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automatique", "Manuelle")
print(tab)
cat("\n")
chisq.test(tab)

## 4. Régression linéaire multiple

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. Diagnostics de régression

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. Intervalles de confiance

In [ ]:
ci <- confint(model, level = 0.95)
cat("Intervalles de confiance à 95 % :\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimation", ylab = "",
     main = "Intervalles de confiance à 95 % pour les coefficients")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. ANOVA à un facteur (One-Way ANOVA)

Le MPG diffère-t-il significativement selon le nombre de cylindres ?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nComparaisons post-hoc de Tukey HSD :\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG par nombre de cylindres",
        xlab = "Cylindres", ylab = "Miles par gallon (MPG)",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## Résumé

- **Test t de Welch** : Dans la comparaison unilatérale non ajustée, les voitures à boîte manuelle affichent une moyenne de MPG plus élevée
- **Chi-deux** : Le tableau de contingence suggère une association, mais les faibles effectifs théoriques déclenchent un avertissement d'approximation, interprétez donc ce résultat avec prudence
- **Régression** : Le poids et la puissance (horsepower) sont des prédicteurs négatifs significatifs après ajustement ; le type de transmission n'est pas significatif dans ce modèle
- **ANOVA** : Le MPG diffère significativement entre les groupes à 4, 6 et 8 cylindres ; les résultats du test de Tukey identifient les différences par paires